In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
from scipy.special import softmax

#Loading in the data sets (white wine and red wine)
white_wine_data = pd.read_csv('winequality-white.csv', delimiter=';')
red_wine_data = pd.read_csv('winequality-red.csv', delimiter=';')

#Adding the bias term to the datasets
white_wine_data['bias'] = 1
red_wine_data['bias'] = 1

#Splitting the data into its features and the target values (quality values)
X_white = white_wine_data.drop('quality', axis=1).values
y_white = white_wine_data['quality'].values
X_red = red_wine_data.drop('quality', axis=1).values
y_red = red_wine_data['quality'].values

#Finding the number of classes (unique quality values)
num_classes = len(np.unique(y_white))

#One-hot encoding the target values
y_white_onehot = np.eye(num_classes)[y_white - 3]  #Here we assume quality values are between 3 and 9
y_red_onehot = np.eye(num_classes)[y_red - 3]

#Training logisitc regression model
def train_logistic_regression(X, y, eta=1e-5, eps = 1e-3, iterations=1000):
    num_samples, num_features = X.shape
    num_classes = y.shape[1]
    weights = np.random.rand(num_features, num_classes)
    old_weights = np.zeros((num_features, num_classes))
    for iteration in range(iterations):
        scores = X.dot(weights)
        probabilities = softmax(scores, axis=1)
        gradient = X.T.dot(probabilities - y) / num_samples
        old_weights = weights
        weights -= eta * gradient
        sum = 0
        for i in range(num_classes):
            difference = weights[i] - old_weights[i]
            sum = sum + np.linalg.norm(difference)
        if sum >= eps:
            return weights
    return weights

#Calling the logisitic regression model functions for each set
weights_white = train_logistic_regression(X_white, y_white_onehot)
weights_red = train_logistic_regression(X_red, y_red_onehot)

#Predicting data using the regression model
def predict(X, weights):
    scores = X.dot(weights)
    probabilities = softmax(scores, axis=1)
    return np.argmax(probabilities, axis=1) + 3

#Calling the predict function for each set
y_pred_white = predict(X_white, weights_white)
y_pred_red = predict(X_red, weights_red)

#Calculates accuracy value
def calculate_accuracy(y_true, y_pred):
    correct = np.sum(y_true == y_pred)
    total = len(y_true)
    accuracy = correct / total
    return accuracy

#Calculates f1 score
def calculate_f1(y_true, y_pred):
    unique_labels = np.unique(y_true)
    f1_scores = []

    for label in unique_labels:
        true_positives = np.sum((y_true == label) * (y_pred == label))
        false_positives = np.sum((y_true != label) * (y_pred == label))
        false_negatives = np.sum((y_true == label) * (y_pred != label))

        if (true_positives + false_positives) != 0: #This if & else statement set prevents division by 0
            precision = true_positives / (true_positives + false_positives)
        else: 
            precision = 0
        recall = true_positives / (true_positives + false_negatives)

        if precision + recall == 0: #Prevents a division by 0 error by appending 0 when the denominator is 0
            f1_scores.append(0)
        else:
            class_f1 = (2 * precision * recall) / (precision + recall)
            if class_f1 > 0: #Prevents a value of "nan" being appended
                f1_scores.append(class_f1)
            else: #Appends a 0 in place of a "nan" value
                f1_scores.append(0)
    
    weights = np.array([np.sum(y_true == label) for label in unique_labels])
    f1_weighted = np.sum(f1_scores * weights) / np.sum(weights)
    return f1_weighted

#Accuracy function from sklearn.metrics and my own for white wine
accuracy_white = calculate_accuracy(y_white, y_pred_white) #This gives the same output as accuracy_score(y_white, y_pred_white)
test_accuracy_white = accuracy_score(y_white, y_pred_white) #Exists for testing purposes

#F1 function from sklearn.metrics and my own for white wine
f1_score_white = calculate_f1(y_white, y_pred_white) #This gives the same output as f1_score(y_white, y_pred_white, average='weighted')
test_f1_score_white = f1_score(y_white, y_pred_white, average='weighted') #Exists for testing purposes

#Accuracy function from sklearn.metrics and my own for red wine
accuracy_red = calculate_accuracy(y_red, y_pred_red) #This gives the same output as accuracy_score(y_red, y_pred_red)
test_accuracy_red = accuracy_score(y_red, y_pred_red) #Exists for testing purposes

#F1 function from sklearn.metrics and my own for red wine
f1_score_red = calculate_f1(y_red, y_pred_red) #This gives the same output as f1_score(y_red, y_pred_red, average='weighted')
test_f1_score_red = f1_score(y_red, y_pred_red, average='weighted') #Exists for testing purposes

#Print data on white wine
print("White Wine Dataset:")
print("Weight Vector:")
print(weights_white)
print("Accuracy:", accuracy_white)
print("F1 Score:", f1_score_white)
print("Test Accuracy with accuracy_score():", test_accuracy_white)
print("Test F1 Score with f1_score():", test_f1_score_white)

#Print data on red wine
print("\nRed Wine Dataset:")
print("Weight Vector:")
print(weights_red)
print("Accuracy:", accuracy_red)
print("F1 Score:", f1_score_red)
print("Test Accuracy with accuracy_score():", test_accuracy_red)
print("Test F1 Score with f1_score():", test_f1_score_red)

White Wine Dataset:
Weight Vector:
[[0.87671452 0.94829005 0.71924516 0.39696463 0.97163753 0.65451637
  0.98287913]
 [0.30092057 0.01919324 0.25623853 0.77589412 0.74752624 0.09381054
  0.74229375]
 [0.86324335 0.63931836 0.802495   0.68840899 0.34291417 0.69886156
  0.6269073 ]
 [0.71477858 0.43093138 0.66582146 0.25138743 0.23735899 0.1580494
  0.87164967]
 [0.22219653 0.53024169 0.25983792 0.52136642 0.68198088 0.68666097
  0.05447705]
 [0.39554882 0.71797196 0.1505575  0.15807025 0.59780402 0.89431295
  0.16467147]
 [0.69350159 0.58336225 0.79559688 0.85497393 0.6379926  0.31659892
  0.7370767 ]
 [0.83272453 0.91888228 0.33164009 0.03897528 0.25036563 0.92911481
  0.55032435]
 [0.52115544 0.77870732 0.9932387  0.14621978 0.10997901 0.12514069
  0.76219357]
 [0.28660268 0.906594   0.51730044 0.1672124  0.61892224 0.36868999
  0.35270523]
 [0.37449031 0.58568785 0.3996864  0.31022552 0.97732428 0.45356522
  0.41066736]
 [0.87096575 0.13766649 0.5733263  0.75706846 0.27724722 0.26118

In [13]:
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from keras import layers

#Loading in the data sets (white wine and red wine)
white_wine_data = pd.read_csv('winequality-white.csv', delimiter=';')
red_wine_data = pd.read_csv('winequality-red.csv', delimiter=';')


X_train = white_wine_data.drop('quality', axis=1)
y_train = white_wine_data['quality']
X_test = red_wine_data.drop('quality', axis=1)
y_test = red_wine_data['quality']

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Creating MLP Model
def create_mlp_model(hidden_layers, hidden_units):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))

    for i in range(hidden_layers):
        model.add(layers.Dense(hidden_units, activation='relu'))

    model.add(layers.Dense(10, activation='softmax'))

    return model

#Looping through different combos of hidden layers and units
best_f1_score = 0
data = []
data_index = 0
best_f1_index = 0

for hidden_layers in [1, 2, 3]:
    for hidden_units in [64, 128, 256]:
        model = create_mlp_model(hidden_layers, hidden_units)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
        y_pred = model.predict(X_test)
        y_pred = np.argmax(y_pred, axis=1)

        f1 = calculate_f1(y_test, y_pred)

        print(f'Hidden Layers: {hidden_layers}, Hidden Units: {hidden_units}, F1-score: {f1}')
        data.append([hidden_layers, hidden_units, f1])
        if f1 > best_f1_score:
            best_f1_score = f1
            best_f1_index = data_index
        data_index = data_index +1

print("\n")
print(f'Best F1-score: {best_f1_score}, Hidden Layers: {data[best_f1_index][0]}, Hidden Units {data[best_f1_index][1]}')


50/50 [==============================] - 0s 2ms/step
Hidden Layers: 1, Hidden Units: 64, F1-score: 0.2574536386619051
50/50 [==============================] - 0s 1ms/step
Hidden Layers: 1, Hidden Units: 128, F1-score: 0.2429722626307403
50/50 [==============================] - 0s 3ms/step
Hidden Layers: 1, Hidden Units: 256, F1-score: 0.1687387434423265
50/50 [==============================] - 0s 1ms/step
Hidden Layers: 2, Hidden Units: 64, F1-score: 0.33400392734463047
50/50 [==============================] - 0s 2ms/step
Hidden Layers: 2, Hidden Units: 128, F1-score: 0.22337232306468782
50/50 [==============================] - 0s 2ms/step
Hidden Layers: 2, Hidden Units: 256, F1-score: 0.28895150012755794
50/50 [==============================] - 0s 2ms/step
Hidden Layers: 3, Hidden Units: 64, F1-score: 0.25925927556958217
50/50 [==============================] - 0s 2ms/step
Hidden Layers: 3, Hidden Units: 128, F1-score: 0.234080428004985
50/50 [==============================] - 0s 2ms/